
# LIVE WEBCAM DRAWING USING OPENCV

## Deskripsi Proyek

Live Webcam Drawing Using OpenCV merupakan proyek visi komputer
yang memungkinkan pengguna menggambar secara virtual menggunakan
gerakan benda berwarna yang terdeteksi oleh webcam.

Program akan mengambil video secara langsung, mendeteksi warna
benda, melacak posisi benda, lalu menggambar jejak pergerakannya
pada sebuah kanvas virtual.

## Tujuan

1. Menerapkan pengolahan video menggunakan OpenCV.
2. Menerapkan deteksi warna untuk mengenali objek.
3. Mendeteksi posisi objek menggunakan contour.
4. Mengimplementasikan gambar virtual berdasarkan gerakan objek.

## Materi Praktikum yang Digunakan

- Prak 2: Pembacaan video menggunakan webcam dan OpenCV.
- Prak 3: Konversi warna dan thresholding untuk memisahkan objek.
- Prak 4: Filtering dan contour detection untuk membantu
  mendeteksi objek.

## Tools dan Library

- Python
- OpenCV
- NumPy
- Jupyter Notebook


## 1. Persiapan Library

Pada tahap ini, kita menggunakan library OpenCV untuk membaca
video dari webcam dan melakukan pemrosesan citra. NumPy digunakan
untuk mengolah data gambar dalam bentuk array, termasuk
menyimpan titik-titik hasil pelacakan objek.


In [1]:

import cv2
import numpy as np

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

OpenCV: 5.0.0
NumPy: 2.5.3



## 2. Pengujian Webcam

Tahap ini bertujuan memastikan webcam dapat diakses oleh program
dan menghasilkan video secara langsung.

Program menggunakan `cv2.VideoCapture(0)` untuk membuka webcam
utama. Setiap frame dibaca menggunakan `cap.read()` dan
ditampilkan menggunakan `cv2.imshow()`.

Pengujian ini merupakan tahap awal sebelum kita menambahkan
deteksi warna dan fitur menggambar secara virtual.

In [2]:

import cv2

cap = cv2.VideoCapture(0)

try:
    if not cap.isOpened():
        print("Webcam tidak dapat dibuka!")

    else:
        print("Webcam berhasil dibuka. Tekan q pada jendela video untuk keluar.")

        while True:
            ret, frame = cap.read()

            if not ret:
                print("Gagal membaca frame dari webcam.")
                break

            cv2.imshow("Tes Webcam", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam ditutup.")

Webcam berhasil dibuka. Tekan q pada jendela video untuk keluar.
Webcam ditutup.



## 3. Thresholding dan Mask Deteksi Multiwarna

Pada tahap ini, program memisahkan objek berdasarkan warna
menggunakan ruang warna HSV dan fungsi `cv2.inRange()`.

Bagian objek yang warnanya sesuai dengan rentang HSV akan
ditampilkan sebagai warna putih, sedangkan bagian yang tidak
sesuai akan ditampilkan sebagai warna hitam.

Setiap warna memiliki mask tersendiri sehingga hasil deteksi
hijau, biru, merah, dan kuning dapat diamati secara terpisah.

In [15]:

import cv2
import numpy as np

cap = cv2.VideoCapture(0)

colors = {
    "Hijau": (np.array([35, 70, 70]), np.array([85, 255, 255])),
    "Biru": (np.array([90, 70, 50]), np.array([130, 255, 255])),
    "Kuning": (np.array([20, 70, 70]), np.array([35, 255, 255])),
    "Merah": (np.array([0, 100, 70]), np.array([10, 255, 255]))
}

try:
    if not cap.isOpened():
        print("Webcam tidak dapat dibuka!")

    else:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Gagal membaca frame.")
                break

            frame = cv2.flip(frame, 1)
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

            # Membuat mask untuk setiap warna
            masks = {}

            for name, (lower, upper) in colors.items():
                masks[name] = cv2.inRange(hsv, lower, upper)

            # Menampilkan video asli
            cv2.imshow("Webcam Asli", frame)

            # Menampilkan hasil thresholding setiap warna
            cv2.imshow("Mask Hijau", masks["Hijau"])
            cv2.imshow("Mask Biru", masks["Biru"])
            cv2.imshow("Mask Kuning", masks["Kuning"])
            cv2.imshow("Mask Merah", masks["Merah"])

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()

## 4. Pelacakan Objek dan Menggambar Virtual

Pada tahap ini, program melacak objek berdasarkan warna yang dipilih pengguna, yaitu merah, hijau, biru, atau kuning. Pemilihan warna dilakukan melalui keyboard sehingga program hanya mendeteksi objek dengan warna yang dipilih.

Setelah warna objek terdeteksi, program mencari contour untuk menentukan posisi objek pada setiap frame webcam. Koordinat pusat objek kemudian digunakan untuk menggambar garis pada kanvas virtual. Lintasan gambar akan mengikuti pergerakan objek yang terdeteksi.

**Petunjuk Penggunaan:**
- Tekan `1` untuk memilih objek berwarna merah.
- Tekan `2` untuk memilih objek berwarna hijau.
- Tekan `3` untuk memilih objek berwarna biru.
- Tekan `4` untuk memilih objek berwarna kuning.
- Tekan `C` untuk menghapus seluruh gambar dan lintasan.
- Tekan `Q` untuk mengakhiri program.

Warna yang sedang dipilih akan ditampilkan pada jendela webcam, sedangkan hasil pemisahan warna dapat diamati melalui jendela Mask Deteksi Warna.

In [16]:

import cv2
import numpy as np
from collections import deque

cap = cv2.VideoCapture(0)

# Rentang warna dalam HSV
colors = {
    "Hijau": [
        (np.array([35, 70, 70]), np.array([85, 255, 255]))
    ],
    "Biru": [
        (np.array([90, 70, 50]), np.array([130, 255, 255]))
    ],
    "Kuning": [
        (np.array([20, 70, 70]), np.array([35, 255, 255]))
    ],
    "Merah": [
        (np.array([0, 70, 50]), np.array([10, 255, 255])),
        (np.array([170, 70, 50]), np.array([180, 255, 255]))
    ]
}

# Pilihan warna berdasarkan tombol keyboard
color_keys = {
    ord("1"): "Merah",
    ord("2"): "Hijau",
    ord("3"): "Biru",
    ord("4"): "Kuning"
}

# Warna awal yang dideteksi
selected_color = "Merah"

# Menyimpan titik-titik lintasan objek
points = deque(maxlen=512)

# Kanvas kosong untuk menggambar
canvas = None

try:
    if not cap.isOpened():
        print("Webcam tidak dapat dibuka!")

    else:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Gagal membaca frame.")
                break

            # Membalik tampilan seperti cermin
            frame = cv2.flip(frame, 1)

            if canvas is None:
                canvas = np.zeros_like(frame)

            # Mengubah BGR menjadi HSV
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

            # Membuat mask berdasarkan warna pilihan
            mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

            for lower, upper in colors[selected_color]:
                mask_part = cv2.inRange(hsv, lower, upper)
                mask = cv2.bitwise_or(mask, mask_part)

            # Mengurangi noise kecil
            kernel = np.ones((5, 5), np.uint8)

            mask = cv2.morphologyEx(
                mask, cv2.MORPH_OPEN, kernel
            )

            mask = cv2.dilate(
                mask, kernel, iterations=1
            )

            # Mencari contour objek
            contours, _ = cv2.findContours(
                mask, cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            # Mengambil objek dengan area terbesar
            if contours:
                largest = max(contours, key=cv2.contourArea)

                if cv2.contourArea(largest) > 300:
                    M = cv2.moments(largest)

                    if M["m00"] != 0:
                        cx = int(M["m10"] / M["m00"])
                        cy = int(M["m01"] / M["m00"])

                        # Menyimpan posisi objek
                        points.appendleft((cx, cy))

                        # Menampilkan titik pusat objek
                        cv2.circle(
                            frame, (cx, cy), 8,
                            (0, 255, 0), -1
                        )
            else:
                # Memutus lintasan jika objek tidak terdeteksi
                if points and points[0] is not None:
                    points.appendleft(None)

            # Menghubungkan titik lintasan pada kanvas
            for i in range(1, len(points)):
                if points[i - 1] is None or points[i] is None:
                    continue

                cv2.line(
                    canvas,
                    points[i - 1],
                    points[i],
                    (0, 255, 0),
                    5
                )

            # Menggabungkan video dan hasil gambar
            result = cv2.add(frame, canvas)

            # Menampilkan warna yang sedang dipilih
            cv2.putText(
                result,
                f"Warna Deteksi: {selected_color}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

            cv2.putText(
                result,
                "1: Merah | 2: Hijau | 3: Biru | 4: Kuning",
                (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 0, 0),
                2
            )

            cv2.imshow("Live Webcam Drawing", result)
            cv2.imshow("Mask Deteksi Warna", mask)

            key = cv2.waitKey(1) & 0xFF

            # Memilih warna yang ingin dideteksi
            if key in color_keys:
                selected_color = color_keys[key]
                points.clear()

            elif key == ord("c"):
                canvas = np.zeros_like(frame)
                points.clear()

            elif key == ord("q"):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()



## 5. Deteksi Ujung Jari Telunjuk

Program menggunakan MediaPipe Hand Landmarker untuk mendeteksi
21 titik penting pada tangan. Landmark nomor 8 merupakan ujung
jari telunjuk.

Koordinat ujung telunjuk digunakan untuk menentukan posisi
kursor gambar pada setiap frame webcam. Pada tahap ini, kita
menguji deteksi ujung telunjuk sebelum menambahkan kanvas
virtual untuk menggambar.

In [17]:

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Lokasi model
model_path = "hand_landmarker.task"

# Pengaturan deteksi tangan untuk video
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)
landmarker = vision.HandLandmarker.create_from_options(options)

frame_timestamp_ms = 0

try:
    if not cap.isOpened():
        print("Webcam tidak dapat dibuka!")

    else:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Gagal membaca frame.")
                break

            frame = cv2.flip(frame, 1)

            # Konversi BGR ke RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Ubah frame menjadi gambar MediaPipe
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            # Deteksi landmark tangan
            result = landmarker.detect_for_video(
                mp_image,
                frame_timestamp_ms
            )
            frame_timestamp_ms += 33

            # Landmark nomor 8 = ujung telunjuk
            if result.hand_landmarks:
                h, w, _ = frame.shape

                for hand in result.hand_landmarks:
                    tip = hand[8]

                    x = int(tip.x * w)
                    y = int(tip.y * h)

                    cv2.circle(
                        frame,
                        (x, y),
                        12,
                        (0, 255, 0),
                        -1
                    )

                    cv2.putText(
                        frame,
                        "Index Finger",
                        (x + 10, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2
                    )

            cv2.imshow("Deteksi Ujung Telunjuk", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

finally:
    cap.release()
    landmarker.close()
    cv2.destroyAllWindows()

## 6. Implementasi Live Webcam Drawing

Pada tahap ini, program menggunakan posisi ujung jari telunjuk sebagai koordinat untuk menggambar pada kanvas virtual. MediaPipe digunakan untuk mendeteksi posisi dan gestur tangan secara langsung melalui webcam.

Program menghubungkan posisi telunjuk dari setiap frame menggunakan fungsi `cv2.line()`, sehingga gerakan telunjuk menghasilkan goresan secara langsung. Hasil gambar ditampilkan pada dua jendela, yaitu tampilan webcam dan kanvas putih agar goresan lebih mudah diamati.

Program juga menyediakan fitur pengaturan ketebalan garis melalui keyboard serta pengenalan gestur tangan untuk memulai dan menghentikan sementara proses menggambar.

**Petunjuk Penggunaan:**

**Gestur tangan:**
- Angkat jari telunjuk untuk mulai menggambar.
- Acungkan ibu jari (*thumbs up*) untuk menghentikan sementara proses menggambar.
- Turunkan atau ubah posisi jari untuk berhenti menggambar sementara.

**Pengaturan ketebalan garis:**
- Tekan `1` untuk garis kecil (3 px).
- Tekan `2` untuk garis sedang (5 px).
- Tekan `3` untuk garis besar (10 px).
- Tekan `4` untuk garis ekstra besar (15 px).

**Kontrol keyboard:**
- Tekan `C` untuk menghapus seluruh gambar.
- Tekan `Q` untuk mengakhiri program.

In [ ]:

import cv2
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Memuat model deteksi tangan
base_options = python.BaseOptions(
    model_asset_path="hand_landmarker.task"
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1
)

landmarker = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

canvas = None
white_canvas = None
previous_point = None
brush_size = 5
timestamp = 0

try:
    if not cap.isOpened():
        print("Webcam tidak dapat dibuka!")

    else:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Gagal membaca frame.")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]

            # Membuat dua kanvas
            if canvas is None:
                canvas = np.zeros_like(frame)
                white_canvas = np.ones_like(frame) * 255

            # Mendeteksi tangan
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp += 33
            result = landmarker.detect_for_video(
                mp_image, timestamp
            )

            status = "Tangan tidak terdeteksi"

            if result.hand_landmarks:
                hand = result.hand_landmarks[0]

                # Koordinat ujung dan ruas jari
                thumb_tip = hand[4]
                thumb_ip = hand[3]

                index_tip = hand[8]
                index_pip = hand[6]

                middle_tip = hand[12]
                middle_pip = hand[10]

                ring_tip = hand[16]
                ring_pip = hand[14]

                pinky_tip = hand[20]
                pinky_pip = hand[18]

                # Mengecek jari selain jempol menekuk
                other_fingers_curled = (
                    index_tip.y > index_pip.y and
                    middle_tip.y > middle_pip.y and
                    ring_tip.y > ring_pip.y and
                    pinky_tip.y > pinky_pip.y
                )

                # Gerakan jempol ke atas = berhenti
                thumbs_up = (
                    thumb_tip.y < thumb_ip.y and
                    other_fingers_curled
                )

                # Telunjuk naik = mulai menggambar
                index_up = (
                    index_tip.y < index_pip.y and
                    middle_tip.y > middle_pip.y and
                    ring_tip.y > ring_pip.y and
                    pinky_tip.y > pinky_pip.y
                )

                if thumbs_up:
                    status = "PAUSE - Jempol"
                    previous_point = None

                elif index_up:
                    status = "DRAWING - Telunjuk"
                    
                    x = int(index_tip.x * width)
                    y = int(index_tip.y * height)
                    current_point = (x, y)

                    # Menggambar pada kedua kanvas
                    if previous_point is not None:

                        # Kanvas webcam
                        cv2.line(canvas, previous_point, current_point, (0,255,0), brush_size)

                        # Kanvas putih
                        cv2.line(white_canvas, previous_point, current_point, (0,180,0), brush_size)

                    previous_point = current_point

                else:
                    status = "Siap - Angkat telunjuk untuk menggambar"
                    previous_point = None

            else:
                previous_point = None

            # Menggabungkan gambar dengan webcam
            output = cv2.add(frame, canvas)

            # Menampilkan bulatan pada ujung telunjuk saat tangan terdeteksi
            if result.hand_landmarks and index_up:
                x = int(index_tip.x * width)
                y = int(index_tip.y * height)

                cv2.circle(output, (x, y), 12, (0, 0, 255), -1)
                cv2.circle(output, (x, y), 16, (255, 255, 255), 2)

            cv2.putText(output, status, (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (0,255,255), 2)

            cv2.putText(output, f"Ukuran Garis: {brush_size}px",
            (10, 60), cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (0, 0, 0), 2)

            cv2.imshow("Live Webcam Drawing", output)
            cv2.imshow("Drawing Canvas", white_canvas)
            key = cv2.waitKey(1) & 0xFF

            # Tombol C untuk menghapus kedua kanvas
            if key == ord("1"):
                brush_size = 3
            elif key == ord("2"):
                brush_size = 5
            elif key == ord("3"):
                brush_size = 10
            elif key == ord("4"):
                brush_size = 15

            elif key == ord("c"):
                canvas = np.zeros_like(frame)
                white_canvas = np.ones_like(frame) * 255
                previous_point = None

            elif key == ord("q"):
                break

finally:
    cap.release()
    landmarker.close()
    cv2.destroyAllWindows()